# MSMCP — End-to-End Evaluation & Stress-Test Notebook

This notebook benchmarks the MSMCP tool surface the way a host LLM reaches it:
it calls the **real registered tool callables**, drives the **real job
executor**, and exercises the **real embedding adapters** in
`src/msmcp/`.

| Section | What it validates |
|---|---|
| 1. Imports & environment | Repo bootstrap, `MSMCP_EMBEDDING_BACKEND`, logging, `tools/list` wire schemas |
| 2. Precursor & adduct suite | `validate_precursor` pass/reject boundaries, `[M+H]+` vs `[M+Na]+` / `[M+2H]2+` / `[M-H]-`, adduct & isotope tables, latency |
| 3. Spectral embeddings | `DreaMSInferenceEmbedder` guards + mock fallbacks, 1024-d shape, determinism, empty/noisy edge cases |
| 4. Async search & polling | Real mzML parsing/QC, `search_library` dispatch, `check_search_status` state transitions, failure / cancellation / concurrency cap |
| 5. Diagnostic summary | Per-tool status, runtime and schema assertions as a Polars/pandas frame |

**Honesty rules the notebook enforces**

* `MSMCP_EMBEDDING_BACKEND=mock` (the default here) selects *deterministic
  dev/test stand-ins* — they are not learned models and must never be reported
  as scientific results. Section 3 asserts the mock label survives all the way
  into tool output.
* `search_library` scans a **synthetic** library seeded from the database path
  string; the library file is **not** opened, while the *query* spectrum is
  genuinely read (from disk or from a server-side reference). Section 4
  asserts both halves of that claim.
* Real inference only runs when the model package is installed **and**
  `MSMCP_EVAL_ALLOW_REAL_INFERENCE=1`; otherwise it is recorded as `SKIP`.

**How to run**

```bash
uv run --with jupyterlab jupyter lab notebooks/eval_msmcp.ipynb
# Polars/pandas are optional; without them the summary renders as Markdown:
# uv add --group dev polars
```

Environment knobs: `MSMCP_EVAL_EMBEDDING_BACKEND` (`mock`|`real`, default
`mock`), `MSMCP_EVAL_ALLOW_REAL_INFERENCE` (`1` to permit real model runs),
`MSMCP_EVAL_LOG_LEVEL` (default `WARNING`), `MSMCP_EVAL_ARTIFACT_DIR`.
Generated mzML fixtures persist in `notebooks/.eval_artifacts/` (gitignored) so
they can be inspected after the run.

## 1. Imports & Environment Setup

Bootstraps `src/` onto `sys.path`, pins the embedding backend **before**
importing any `msmcp.models` module, configures stderr logging, and captures
the real tool callables by replaying `register_tools()` against a minimal
MCPServer stand-in (the same pattern `tests/conftest.py` uses). Wire schemas
are read from the *real* `MCPServer` via `tools/list`.

In [ ]:
# =============================================================================
# 1.1 Repo bootstrap & environment flags
# =============================================================================
import asyncio
import base64
import importlib.util
import json
import logging
import os
import platform
import re
import sys
import tempfile
import threading
import time
import traceback
import uuid
from collections import defaultdict
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
NOTEBOOK_DIR = CWD if CWD.name == "notebooks" else CWD / "notebooks"
SRC_DIR = REPO_ROOT / "src"
if not (SRC_DIR / "msmcp" / "__init__.py").exists():
    raise RuntimeError(f"could not locate the msmcp source tree under {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ---------------------------------------------------------------------------
# Embedding backend flag.  MUST be set before msmcp.models is imported so the
# resolver sees it.  "real" = strict production inference; "mock" = the
# deterministic dev/test stand-ins (no torch, no checkpoint download).
# ---------------------------------------------------------------------------
EVAL_BACKEND = os.environ.get("MSMCP_EVAL_EMBEDDING_BACKEND", "mock").lower()
if EVAL_BACKEND not in ("mock", "real"):
    raise ValueError(
        f"MSMCP_EVAL_EMBEDDING_BACKEND must be 'mock' or 'real', got {EVAL_BACKEND!r}"
    )
os.environ["MSMCP_EMBEDDING_BACKEND"] = EVAL_BACKEND

ALLOW_REAL_INFERENCE = os.environ.get("MSMCP_EVAL_ALLOW_REAL_INFERENCE") == "1"
LOG_LEVEL = os.environ.get("MSMCP_EVAL_LOG_LEVEL", "WARNING").upper()

# Artifacts live inside notebooks/ so the default MSMCP security root (the
# process working directory) always contains them; the file-backed tools are
# exercised on these fixtures in Section 4.
ARTIFACT_DIR = (
    Path(os.environ.get("MSMCP_EVAL_ARTIFACT_DIR", NOTEBOOK_DIR / ".eval_artifacts"))
    .expanduser()
    .resolve()
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=getattr(logging, LOG_LEVEL, logging.WARNING),
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    stream=sys.stderr,
    force=True,
)
logger = logging.getLogger(
    "msmcp.eval"
)  # available for ad-hoc debugging in later cells

# ---------------------------------------------------------------------------
# Imports: tool modules, engine adapters, security boundary, executor internals
# ---------------------------------------------------------------------------
from pydantic import ValidationError

import msmcp
from msmcp.errors import (
    InaccessiblePathError,
    MalformedFileError,
    MsmcpError,
    SpectrumIndexError,
)
from msmcp.execution import UnknownJobError
from msmcp.models import (
    DreaMSEmbedder,
    DreaMSInferenceEmbedder,
    EmbeddingBackendUnavailable,
    LSMMS2Embedder,
    LSMMS2InferenceEmbedder,
    get_embedder,
)
from msmcp.models.embeddings import EMBEDDING_DIM, MZ_SPAN
from msmcp.security import DEFAULT_POLICY, PathEscapeError
from msmcp.state.pointers import UnknownReferenceError
from msmcp.tools import chem, qc, search, similarity
from msmcp.tools import io as msmcp_io
from msmcp.tools.search import _EXECUTOR

PACKAGE_PATH = Path(msmcp.__file__).resolve()
SUITE_START = time.perf_counter()

print(f"msmcp package      : {PACKAGE_PATH}")
print(f"python             : {platform.python_version()} ({platform.system()})")
print(f"numpy              : {np.__version__}")
print(f"embedding backend  : {os.environ['MSMCP_EMBEDDING_BACKEND']!r}")
print(f"real inference ok? : {ALLOW_REAL_INFERENCE}")
print(f"log level          : {LOG_LEVEL}")
print(f"allowed root       : {DEFAULT_POLICY.allowed_root}")
print(f"artifact dir       : {ARTIFACT_DIR}")

In [ ]:
# =============================================================================
# 1.2 Tool registry — capture the real callables, not reimplementations
# =============================================================================
class ToolCapture:
    """Minimal MCPServer stand-in that records the tools registered by a module.

    ``register_tools(mcp)`` only ever calls ``mcp.tool(**metadata)`` as a
    decorator, so this two-method shim yields the exact callables the server
    publishes, without going through the JSON-RPC layer.
    """

    def __init__(self) -> None:
        self.tools: dict[str, Callable[..., Any]] = {}

    def tool(
        self, **_metadata: Any
    ) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
        def decorator(fn: Callable[..., Any]) -> Callable[..., Any]:
            self.tools[fn.__name__] = fn
            return fn

        return decorator


_capture = ToolCapture()
for _module in (chem, msmcp_io, qc, search, similarity):
    _module.register_tools(_capture)
TOOLS = _capture.tools
ASYNC_TOOLS = {name for name, fn in TOOLS.items() if asyncio.iscoroutinefunction(fn)}

EXPECTED_TOOLS = {
    "annotate_isotopes",
    "cancel_search",
    "check_search_status",
    "compute_cosine",
    "generate_qc_summary",
    "load_spectrum",
    "load_mzml_summary",
    "release_reference",
    "summarise_reference",
    "predict_adduct_offset",
    "search_library",
    "validate_precursor",
}
_missing_tools = EXPECTED_TOOLS - set(TOOLS)
assert not _missing_tools, f"registered tool drift — missing: {sorted(_missing_tools)}"
print(f"captured {len(TOOLS)} tools | async: {sorted(ASYNC_TOOLS)}")
print("extras beyond the expected set:", sorted(set(TOOLS) - EXPECTED_TOOLS) or "none")

In [ ]:
# =============================================================================
# 1.3 Async bridge — one private event loop for every async tool call
# =============================================================================
# All async work runs on a single background loop so module-level primitives
# (e.g. the search semaphore) bind to one consistent loop and individual cells
# stay re-runnable.
class AsyncRunner:
    """Drive coroutines from synchronous notebook cells on a private loop."""

    def __init__(self) -> None:
        self._loop = asyncio.new_event_loop()
        self._thread = threading.Thread(
            target=self._run_forever, name="msmcp-eval-loop", daemon=True
        )
        self._thread.start()

    def _run_forever(self) -> None:
        asyncio.set_event_loop(self._loop)
        self._loop.run_forever()

    def run(self, coro: Any, timeout: float = 600.0) -> Any:
        return asyncio.run_coroutine_threadsafe(coro, self._loop).result(
            timeout=timeout
        )


RUNNER = AsyncRunner()


def sync_call(name: str, /, **kwargs: Any) -> Any:
    """Call a synchronous tool by name."""
    return TOOLS[name](**kwargs)


def async_call(name: str, /, timeout: float = 600.0, **kwargs: Any) -> Any:
    """Call an async tool by name and block until it returns."""
    return RUNNER.run(TOOLS[name](**kwargs), timeout=timeout)


print("async runner ready on", RUNNER._loop)

In [ ]:
# =============================================================================
# 1.4 Check harness — every assertion becomes a row in the summary table
# =============================================================================
@dataclass
class CheckResult:
    """One evaluated assertion, rendered in the Section 5 diagnostic table."""

    tool: str
    case: str
    input_status: str
    status: str  # PASS | FAIL | SKIP
    runtime_ms: float
    schema_assertion: str
    detail: str = ""


RESULTS: list[CheckResult] = []
_ICON = {"PASS": "✅", "FAIL": "❌", "SKIP": "⏭️"}


def expect(condition: Any, message: str) -> None:
    """Assertion helper small enough to use inside a lambda predicate."""
    if not condition:
        raise AssertionError(message)


def must_contain(text: Any, *needles: str) -> str:
    """Assertion helper: every needle appears in the text."""
    expect(isinstance(text, str), f"expected text, got {type(text).__name__}")
    for needle in needles:
        expect(needle in text, f"missing {needle!r} in output:\n{text[:800]}")
    return text


def _record(row: CheckResult) -> None:
    RESULTS.append(row)
    print(
        f"{_ICON.get(row.status, '?')} [{row.tool}] {row.case} — {row.status} "
        f"({row.runtime_ms:.2f} ms) :: {row.schema_assertion}"
    )
    if row.status == "FAIL" and row.detail:
        print("    " + row.detail.replace("\n", "\n    ")[:1200])


def check(
    tool: str,
    case: str,
    call: Callable[[], Any],
    *,
    predicate: Callable[[Any], Any] | None = None,
    raises: type[BaseException] | tuple[type[BaseException], ...] | None = None,
    input_status: str = "-",
    assertion: str = "-",
    note: str = "",
) -> Any:
    """Time *call*, verify it (or the exception it raises), and record a row.

    Never raises: an unexpected exception is recorded as FAIL so the notebook
    keeps evaluating the remaining cases. Returns the value (or exception).
    """
    started = time.perf_counter()
    try:
        value, exc, exc_tb = call(), None, ""
    except Exception as caught:
        value, exc = None, caught
        exc_tb = traceback.format_exc()
    runtime_ms = (time.perf_counter() - started) * 1000.0

    if raises is not None:
        if exc is None:
            _record(
                CheckResult(
                    tool,
                    case,
                    input_status,
                    "FAIL",
                    runtime_ms,
                    assertion,
                    f"expected {raises!r}, but the call returned {value!r}",
                )
            )
            return value
        if not isinstance(exc, raises):
            _record(
                CheckResult(
                    tool,
                    case,
                    input_status,
                    "FAIL",
                    runtime_ms,
                    assertion,
                    f"expected {raises!r}, got {type(exc).__name__}: {exc}\n{exc_tb}",
                )
            )
            return exc
        if predicate is not None:
            try:
                predicate(exc)
            except AssertionError as failed:
                _record(
                    CheckResult(
                        tool,
                        case,
                        input_status,
                        "FAIL",
                        runtime_ms,
                        assertion or f"raised {type(exc).__name__}",
                        str(failed),
                    )
                )
                return exc
        _record(
            CheckResult(
                tool,
                case,
                input_status,
                "PASS",
                runtime_ms,
                assertion or f"raised {type(exc).__name__} as expected",
                note,
            )
        )
        return exc

    if exc is not None:
        _record(
            CheckResult(
                tool,
                case,
                input_status,
                "FAIL",
                runtime_ms,
                assertion or "call must not raise",
                f"{exc_tb}",
            )
        )
        return None
    if predicate is not None:
        try:
            predicate(value)
        except AssertionError as failed:
            _record(
                CheckResult(
                    tool, case, input_status, "FAIL", runtime_ms, assertion, str(failed)
                )
            )
            return value
    _record(CheckResult(tool, case, input_status, "PASS", runtime_ms, assertion, note))
    return value


def skip(tool: str, case: str, reason: str, *, input_status: str = "-") -> None:
    _record(CheckResult(tool, case, input_status, "SKIP", 0.0, reason))


print("harness ready")

In [ ]:
# =============================================================================
# 1.5 Wire schema (tools/list) — the surface a host LLM actually sees
# =============================================================================
MODEL_FOR_TOOL: dict[str, type[Any]] = {
    "load_mzml_summary": msmcp_io.MzMLParseInput,
    "load_spectrum": msmcp_io.LoadSpectrumInput,
    "summarise_reference": msmcp_io.ReferenceInput,
    "release_reference": msmcp_io.ReferenceInput,
    "generate_qc_summary": qc.QCInput,
    "predict_adduct_offset": chem.AdductInput,
    "annotate_isotopes": chem.IsotopeInput,
    "validate_precursor": similarity.ValidatePrecursorInput,
    "compute_cosine": similarity.ComputeCosineInput,
    "search_library": search.SearchInput,
    "check_search_status": search.PollInput,
    "cancel_search": search.StatusInput,
}

try:
    from msmcp.server import mcp as REAL_SERVER

    WIRE_SCHEMAS = {
        tool.name: tool for tool in RUNNER.run(REAL_SERVER.list_tools(), timeout=60)
    }
    SERVER_IMPORT_ERROR = ""
except Exception as exc:  # surfaces a broken environment as a failed row
    REAL_SERVER, WIRE_SCHEMAS = None, {}
    SERVER_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"

print(f"tools/list returned {len(WIRE_SCHEMAS)} tools")
print("server import error:", SERVER_IMPORT_ERROR or "none")


def wire_probe(tool_name: str) -> Callable[[], str]:
    """Return a probe asserting one tool's wire contract."""

    def probe() -> str:
        tool = WIRE_SCHEMAS[tool_name]
        spec = tool.input_schema
        props = spec.get("properties", {})

        undocumented = [p for p, s in props.items() if not s.get("description")]
        expect(not undocumented, f"parameters with no wire description: {undocumented}")

        model = MODEL_FOR_TOOL.get(tool_name)
        if model is not None:
            expect(
                set(props) == set(model.model_fields),
                f"signature/model drift: wire={sorted(props)} model={sorted(model.model_fields)}",
            )
            required = {n for n, f in model.model_fields.items() if f.is_required()}
            expect(
                set(spec.get("required", [])) == required,
                f"required drift: wire={spec.get('required')} model={sorted(required)}",
            )

        annotations = tool.annotations
        expect(annotations is not None, "no annotations")
        expect(annotations.read_only_hint is not None, "read_only_hint unset")
        expect(annotations.open_world_hint is not None, "open_world_hint unset")
        expect(bool(tool.title and tool.description), "missing title/description")
        return f"{len(props)} params, required={sorted(spec.get('required', []))}"

    return probe


check(
    "tools/list",
    "real MCPServer publishes the tool surface",
    lambda: sorted(WIRE_SCHEMAS),
    predicate=lambda names: expect(
        set(names) >= EXPECTED_TOOLS,
        f"missing from tools/list: {sorted(EXPECTED_TOOLS - set(names))}",
    ),
    input_status="msmcp.server.mcp (all modules registered)",
    assertion=f"tools/list reachable; every expected tool published (n={len(EXPECTED_TOOLS)})",
    note=SERVER_IMPORT_ERROR,
)

for _name in sorted(WIRE_SCHEMAS):
    check(
        "tools/list",
        f"wire schema: {_name}",
        wire_probe(_name),
        input_status="tools/list schema",
        assertion="params documented; required set matches the validating Pydantic model; annotations present",
    )

writing_tools = sorted(
    name
    for name, tool in WIRE_SCHEMAS.items()
    if tool.annotations is not None and tool.annotations.read_only_hint is False
)
check(
    "tools/list",
    "only cancel_search and release_reference mutate server state",
    lambda: writing_tools,
    predicate=lambda names: expect(
        names == ["cancel_search", "release_reference"],
        f"unexpected non-read-only tools: {names}",
    ),
    input_status="tools/list annotations",
    assertion="non-read-only tools are exactly cancel_search and release_reference",
)

## 2. Precursor & Adduct Validation Suite

`validate_precursor` compares an observed precursor m/z with a hypothesised
exact mass and passes only at **≤ 5.0 ppm**.

* Boundary cases at 0 / ±2.5 / ±4.9 ppm (pass) and 5.1 / 12 / −9 ppm (reject).
* Adduct shifts come from `predict_adduct_offset` (cross-checked against the
  module's `_ADDUCT_DB`) and are then fed back into `validate_precursor` for
  `[M+H]+`, `[M+Na]+`, `[M+2H]2+` and `[M-H]-`.
* The classic mis-assignment trap: validating the *neutral* mass against a
  sodium-adduct observation must be rejected.
* `annotate_isotopes` is cross-checked against the same exact mass.
* Pydantic bounds (`gt=0`) must raise before any chemistry runs.
* Every case records wall-clock latency.

In [ ]:
# =============================================================================
# 2.1 Reference data & report parsers
# =============================================================================
CAFFEINE = {"name": "Caffeine", "formula": "C8H10N4O2", "mono": 194.080375574}

_PPM_RE = re.compile(r"Mass error:\s+([0-9.]+) ppm")
_SHIFT_RE = re.compile(r"Exact mass shift \(Δ\): ([+-]?[0-9.]+) Da")
_CHARGE_RE = re.compile(r"Charge state: ([+-]?[0-9]+)")
_MONO_RE = re.compile(r"Monoisotopic mass: \*\*([0-9.]+) Da\*\*")


def parse_ppm(text: str) -> float:
    match = _PPM_RE.search(text)
    if match is None:
        raise AssertionError(f"no ppm value in output:\n{text}")
    return float(match.group(1))


def parse_shift(text: str) -> tuple[float, int]:
    shift, charge = _SHIFT_RE.search(text), _CHARGE_RE.search(text)
    if shift is None or charge is None:
        raise AssertionError(f"unparseable adduct report:\n{text}")
    return float(shift.group(1)), int(charge.group(1))


def parse_mono_mass(text: str) -> float:
    match = _MONO_RE.search(text)
    if match is None:
        raise AssertionError(f"no monoisotopic mass in output:\n{text}")
    return float(match.group(1))


def shift_by_adduct(adduct: str) -> tuple[float, int, str]:
    """Ask the tool for an adduct shift (its own report is the source of truth)."""
    text = sync_call("predict_adduct_offset", adduct_string=adduct)
    delta, charge = parse_shift(text)
    return delta, charge, text


def mz_for(neutral_mass: float, adduct: str) -> float:
    """Theoretical m/z of *neutral_mass* under *adduct* (Δ already charge-scaled)."""
    delta, charge, _ = shift_by_adduct(adduct)
    return (neutral_mass + delta) / abs(charge)


def expected_ppm(theoretical: float, experimental: float) -> float:
    return abs(theoretical - experimental) / theoretical * 1e6


print(f"{CAFFEINE['name']} ({CAFFEINE['formula']}) monoisotopic: {CAFFEINE['mono']} Da")
for _adduct in ("[M+H]+", "[M+Na]+", "[M+2H]2+", "[M-H]-"):
    _delta, _charge, _ = shift_by_adduct(_adduct)
    print(
        f"  {_adduct:<10} Δ = {_delta:+.6f} Da (z={_charge:+d}) "
        f"→ m/z {mz_for(CAFFEINE['mono'], _adduct):.4f}"
    )

In [ ]:
# =============================================================================
# 2.2 Reference-table integrity — adduct shifts and isotope patterns
# =============================================================================
_ADDUCTS = (
    "[M+H]+",
    "[M+Na]+",
    "[M+K]+",
    "[M+NH4]+",
    "[M+2H]2+",
    "[M-H]-",
    "[M+Cl]-",
    "[M+HCOO]-",
)
for _adduct in _ADDUCTS:
    check(
        "predict_adduct_offset",
        f"shift for {_adduct}",
        lambda a=_adduct: sync_call("predict_adduct_offset", adduct_string=a),
        predicate=lambda text, a=_adduct: expect(
            abs(parse_shift(text)[0] - chem._ADDUCT_DB[a]["shift"]) < 1e-6,
            f"{a}: tool shift {parse_shift(text)[0]} != database {chem._ADDUCT_DB[a]['shift']}",
        ),
        input_status=f"adduct string {_adduct}",
        assertion="report shift equals _ADDUCT_DB shift; charge and polarity present",
    )

for _bad in ("[M+Li]+", "[M+2]+", "M+H"):
    check(
        "predict_adduct_offset",
        f"hallucinated adduct {_bad!r} is rejected",
        lambda b=_bad: sync_call("predict_adduct_offset", adduct_string=b),
        predicate=lambda text: must_contain(text, "REJECTED", "Supported adducts are:"),
        input_status=f"invalid adduct {_bad!r}",
        assertion="REJECTED verdict listing the supported adducts",
    )

check(
    "annotate_isotopes",
    f"caffeine ({CAFFEINE['formula']}) isotope pattern agrees with the exact mass",
    lambda: sync_call("annotate_isotopes", identifier=CAFFEINE["formula"]),
    predicate=lambda text: (
        must_contain(
            text,
            f"## Isotope Pattern: {CAFFEINE['formula']}",
            "Monoisotopic mass: **194.0804 Da**",
            "| M+1",
            "| M+2",
        ),
        expect(
            abs(parse_mono_mass(text) - CAFFEINE["mono"]) < 1e-4,
            "isotope tool disagrees with the reference monoisotopic mass",
        ),
    )[-1],
    input_status=f"identifier={CAFFEINE['formula']}",
    assertion="M/M+1/M+2 table; monoisotopic mass matches the validate_precursor reference",
)

In [ ]:
# =============================================================================
# 2.3 Mass-error boundary suite (5.0 ppm acceptance threshold)
# =============================================================================
_PASS_ERRORS = [
    ("exact match (0 ppm)", 0.0),
    ("+2.5 ppm within tolerance", 2.5),
    ("+4.9 ppm at the tolerance edge", 4.9),
    ("-4.9 ppm negative drift at the edge", -4.9),
]
_REJECT_ERRORS = [
    ("+5.1 ppm just outside tolerance", 5.1),
    ("+12 ppm miscalibrated instrument", 12.0),
    ("-9 ppm negative miscalibration", -9.0),
    ("+118450 ppm (~12%) wrong species", 118_450.0),
]


def precursor_case(theoretical: float, error_ppm: float) -> float:
    """Observed m/z that sits *error_ppm* away from the theoretical mass."""
    return theoretical * (1.0 + error_ppm * 1e-6)


for _label, _error in _PASS_ERRORS:
    _observed = precursor_case(CAFFEINE["mono"], _error)
    check(
        "validate_precursor",
        _label,
        lambda t=CAFFEINE["mono"], o=_observed: sync_call(
            "validate_precursor", theoretical_mass=t, experimental_mass=o
        ),
        predicate=lambda text, e=_error: (
            must_contain(text, "VALIDATION PASSED", "≤ 5.0 ppm threshold"),
            expect(
                abs(parse_ppm(text) - abs(e)) < 0.05,
                f"reported {parse_ppm(text)} ppm, expected {abs(e)}",
            ),
        )[-1],
        input_status=f"C8H10N4O2 exact mass, {_error:+.1f} ppm error",
        assertion="VALIDATION PASSED; reported ppm within 0.05 of the injected error",
    )

for _label, _error in _REJECT_ERRORS:
    _observed = precursor_case(CAFFEINE["mono"], _error)
    check(
        "validate_precursor",
        _label,
        lambda t=CAFFEINE["mono"], o=_observed: sync_call(
            "validate_precursor", theoretical_mass=t, experimental_mass=o
        ),
        predicate=lambda text, e=_error: (
            must_contain(
                text, "VALIDATION REJECTED", "exceeds the 5.0 ppm", "physically invalid"
            ),
            expect(
                abs(parse_ppm(text) - abs(e)) / abs(e) < 1e-3,
                f"reported {parse_ppm(text)} ppm, expected {abs(e)}",
            ),
        )[-1],
        input_status=f"C8H10N4O2 exact mass, {_error:+.1f} ppm error",
        assertion="VALIDATION REJECTED; reported ppm consistent with the injected error",
    )

In [ ]:
# =============================================================================
# 2.4 Adduct-aware precursor validation (positive & negative mode)
# =============================================================================
_ADDUCT_CASES = [
    ("[M+H]+ protonated, +1.5 ppm", "[M+H]+", 1.5),
    ("[M+Na]+ sodiated, -2.0 ppm", "[M+Na]+", -2.0),
    ("[M+2H]2+ doubly charged, +0.8 ppm", "[M+2H]2+", 0.8),
    ("[M-H]- deprotonated, -1.0 ppm", "[M-H]-", -1.0),
]

for _label, _adduct, _error in _ADDUCT_CASES:
    _theoretical = mz_for(CAFFEINE["mono"], _adduct)
    _observed = precursor_case(_theoretical, _error)
    check(
        "validate_precursor",
        _label,
        lambda t=_theoretical, o=_observed: sync_call(
            "validate_precursor", theoretical_mass=t, experimental_mass=o
        ),
        predicate=lambda text, e=_error: (
            must_contain(text, "VALIDATION PASSED"),
            expect(
                abs(parse_ppm(text) - abs(e)) < 0.05,
                f"reported {parse_ppm(text)} ppm, expected {abs(e)}",
            ),
        )[-1],
        input_status=f"theoretical m/z {_theoretical:.4f} ({_adduct}), {_error:+.1f} ppm",
        assertion="adduct-specific m/z passes (shift and charge handled correctly)",
    )

# The classic mis-assignment: neutral caffeine validated against the [M+Na]+ ion.
_sodiated_mz = mz_for(CAFFEINE["mono"], "[M+Na]+")
_misassignment_ppm = expected_ppm(CAFFEINE["mono"], _sodiated_mz)
check(
    "validate_precursor",
    "neutral mass vs [M+Na]+ observation is rejected",
    lambda: sync_call(
        "validate_precursor",
        theoretical_mass=CAFFEINE["mono"],
        experimental_mass=_sodiated_mz,
    ),
    predicate=lambda text: (
        must_contain(text, "VALIDATION REJECTED"),
        expect(
            abs(parse_ppm(text) - _misassignment_ppm) < 1.0,
            f"expected ~{_misassignment_ppm:.0f} ppm, got {parse_ppm(text):.1f} ppm",
        ),
    )[-1],
    input_status=f"neutral {CAFFEINE['mono']:.4f} vs [M+Na]+ m/z {_sodiated_mz:.4f}",
    assertion=f"adduct mis-assignment caught: {_misassignment_ppm:,.0f} ppm error > 5 ppm",
)

In [ ]:
# =============================================================================
# 2.5 Schema guards — invalid masses must be rejected before any chemistry
# =============================================================================
check(
    "validate_precursor",
    "zero theoretical mass is rejected by the schema",
    lambda: sync_call(
        "validate_precursor", theoretical_mass=0.0, experimental_mass=100.0
    ),
    raises=ValidationError,
    input_status="theoretical_mass=0.0",
    assertion="Pydantic gt=0 bound enforced (ValidationError)",
)

check(
    "validate_precursor",
    "negative experimental mass is rejected by the schema",
    lambda: sync_call(
        "validate_precursor", theoretical_mass=100.0, experimental_mass=-1.0
    ),
    raises=ValidationError,
    input_status="experimental_mass=-1.0",
    assertion="Pydantic gt=0 bound enforced (ValidationError)",
)

check(
    "validate_precursor",
    "non-numeric mass is rejected by the schema",
    lambda: sync_call(
        "validate_precursor", theoretical_mass="caffeine", experimental_mass=100.0
    ),
    raises=ValidationError,
    input_status='theoretical_mass="caffeine"',
    assertion="type coercion refuses a non-numeric mass (ValidationError)",
)

check(
    "predict_adduct_offset",
    "too-short adduct string is rejected by the schema",
    lambda: sync_call("predict_adduct_offset", adduct_string="ab"),
    raises=ValidationError,
    input_status='adduct_string="ab"',
    assertion="Pydantic min_length=3 enforced (ValidationError)",
)

_p2_rows = [
    r
    for r in RESULTS
    if r.tool in ("validate_precursor", "predict_adduct_offset", "annotate_isotopes")
]
print(
    f"\nSection 2: {len(_p2_rows)} cases | "
    f"PASS {sum(r.status == 'PASS' for r in _p2_rows)} | "
    f"FAIL {sum(r.status == 'FAIL' for r in _p2_rows)} | "
    f"median latency {float(np.median([r.runtime_ms for r in _p2_rows])):.3f} ms"
)

## 3. Spectral Embedding & Model Inference

Exercises the embedding contract with `(N, 2)` peak arrays:

* `DreaMSInferenceEmbedder` — the real-inference adapter. Its *input guards*
  (empty peak list, all peaks at/above the precursor) raise before any model
  load, so they are testable without the `dreams` package or a checkpoint. A
  full inference pass runs only when the package is installed **and**
  `MSMCP_EVAL_ALLOW_REAL_INFERENCE=1`.
* `DreaMSEmbedder` / `LSMMS2Embedder` — the deterministic dev/test stand-ins
  selected by `MSMCP_EMBEDDING_BACKEND=mock`, used as the hermetic fallback
  that proves the 1024-d pipeline, determinism and label plumbing.
* Degenerate inputs: empty arrays, zero intensities, a single peak, peaks
  above the projection span, and m/z ≥ precursor.

In [ ]:
# =============================================================================
# 3.1 Sample spectra
# =============================================================================
@dataclass(frozen=True)
class SpectrumCase:
    name: str
    precursor_mz: float
    peaks: np.ndarray
    note: str = ""


def peaks(*pairs: tuple[float, float]) -> np.ndarray:
    return np.asarray(pairs, dtype=np.float64)


CAFFEINE_MS2 = SpectrumCase(
    "caffeine MS2",
    194.0804,
    peaks(
        (138.0662, 4.20e5),
        (110.0713, 2.15e5),
        (123.0426, 1.40e5),
        (83.0604, 9.00e4),
        (69.0447, 7.50e4),
        (55.0291, 6.20e4),
        (42.0338, 4.40e4),
        (152.0819, 3.10e4),
        (95.0604, 2.40e4),
    ),
    "caffeine fragment ions",
)
GLUCOSE_MS2 = SpectrumCase(
    "glucose MS2",
    180.0634,
    peaks(
        (163.0601, 5.10e5),
        (145.0495, 3.30e5),
        (127.0390, 2.20e5),
        (85.0289, 1.60e5),
        (73.0289, 1.10e5),
        (60.0211, 8.00e4),
    ),
    "hexose fragment ions",
)
NOISY_CAFFEINE = SpectrumCase(
    "caffeine MS2 + 3% baseline noise",
    CAFFEINE_MS2.precursor_mz,
    np.vstack(
        [
            CAFFEINE_MS2.peaks,
            peaks(
                (201.0871, 1.26e4),
                (176.0700, 1.10e4),
                (160.0752, 9.00e3),
                (130.0651, 7.00e3),
                (102.0337, 6.00e3),
                (88.0393, 5.00e3),
                (74.0237, 4.00e3),
            ),
        ]
    ),
    "same fragments plus a low-intensity background",
)
SINGLE_PEAK = SpectrumCase("single peak", 120.0, peaks((100.0, 1.0)))
ZERO_INTENSITY = SpectrumCase(
    "all-zero intensities", 150.0, peaks((100.0, 0.0), (120.0, 0.0))
)
EMPTY = SpectrumCase("empty peak array", 150.0, np.zeros((0, 2), dtype=np.float64))
ABOVE_SPAN = SpectrumCase(
    "peaks above the 2048 Da projection span",
    2500.0,
    peaks((2100.0, 5.0e4), (2400.0, 3.0e4)),
)
ABOVE_PRECURSOR = SpectrumCase(
    "all peaks at/above the precursor",
    100.0,
    peaks((100.0, 5.0e4), (140.0, 3.0e4)),
)

print(f"embedding dim: {EMBEDDING_DIM} | m/z span: {MZ_SPAN} Da")
for case in (CAFFEINE_MS2, GLUCOSE_MS2, NOISY_CAFFEINE):
    print(
        f"  {case.name:<32} {len(case.peaks):>3} peaks, precursor {case.precursor_mz:.4f}"
    )

In [ ]:
# =============================================================================
# 3.2 Mock fallback adapters — shape, dtype, normalisation, determinism
# =============================================================================
def vector_ok(vector: np.ndarray, dim: int = EMBEDDING_DIM) -> None:
    expect(isinstance(vector, np.ndarray), f"not an ndarray: {type(vector)}")
    expect(vector.shape == (dim,), f"shape {vector.shape} != ({dim},)")
    expect(vector.dtype == np.float32, f"dtype {vector.dtype} != float32")
    expect(bool(np.isfinite(vector).all()), "non-finite values in the embedding")
    norm = float(np.linalg.norm(vector))
    expect(abs(norm - 1.0) < 1e-5, f"not L2-normalised: norm={norm}")


MOCK_EMBEDDERS = {"dreams": DreaMSEmbedder(), "lsm-ms2": LSMMS2Embedder()}

for _method, _embedder in MOCK_EMBEDDERS.items():
    check(
        "embed",
        f"{_method}: 1024-d unit vector for {CAFFEINE_MS2.name}",
        lambda e=_embedder: e.embed_spectrum(
            CAFFEINE_MS2.peaks, precursor_mz=CAFFEINE_MS2.precursor_mz
        ),
        predicate=vector_ok,
        input_status=f"{len(CAFFEINE_MS2.peaks)} peaks, precursor {CAFFEINE_MS2.precursor_mz}",
        assertion="shape (1024,), float32, finite, L2 norm == 1",
    )

    check(
        "embed",
        f"{_method}: deterministic across repeated calls",
        lambda e=_embedder: np.array_equal(
            e.embed_spectrum(
                CAFFEINE_MS2.peaks, precursor_mz=CAFFEINE_MS2.precursor_mz
            ),
            e.embed_spectrum(
                CAFFEINE_MS2.peaks, precursor_mz=CAFFEINE_MS2.precursor_mz
            ),
        ),
        predicate=lambda same: expect(
            same, "two embeddings of the same peak list differ"
        ),
        input_status="identical peak arrays, two calls",
        assertion="byte-identical vectors (deterministic seed, no hidden state)",
    )

    check(
        "embed",
        f"{_method}: peak-order invariant",
        lambda e=_embedder: float(
            np.dot(
                e.embed_spectrum(
                    CAFFEINE_MS2.peaks, precursor_mz=CAFFEINE_MS2.precursor_mz
                ),
                e.embed_spectrum(
                    CAFFEINE_MS2.peaks[::-1], precursor_mz=CAFFEINE_MS2.precursor_mz
                ),
            )
        ),
        predicate=lambda cosine: expect(
            abs(cosine - 1.0) < 1e-6,
            f"reversed peak order changes the embedding (cos={cosine})",
        ),
        input_status="same peaks in reverse order",
        assertion="cosine == 1.0 (the seed hashes m/z-sorted peaks)",
    )

    check(
        "embed",
        f"{_method}: fails loudly on an empty spectrum",
        lambda e=_embedder: e.embed_spectrum(EMPTY.peaks),
        raises=ValueError,
        predicate=lambda exc: must_contain(str(exc), "non-empty (N, 2)"),
        input_status="shape (0, 2) array",
        assertion="ValueError naming the required non-empty (N, 2) shape",
    )

# Cross-model separation: each adapter uses a distinct salt and intensity
# compression, so the two embedding spaces must not collapse onto each other.
_cross = float(
    np.dot(
        MOCK_EMBEDDERS["dreams"].embed_spectrum(CAFFEINE_MS2.peaks),
        MOCK_EMBEDDERS["lsm-ms2"].embed_spectrum(CAFFEINE_MS2.peaks),
    )
)
check(
    "embed",
    "DreaMS vs LSM-MS2 mock spaces are distinct",
    lambda: _cross,
    predicate=lambda c: expect(c < 0.99, f"model spaces collapsed (cos={c})"),
    input_status="same caffeine spectrum through both adapters",
    assertion=f"cross-model cosine {_cross:.4f} < 0.99 (distinct salts)",
)

In [ ]:
# =============================================================================
# 3.3 Degenerate spectra — the edge cases a noisy acquisition produces
# =============================================================================
for _case in (SINGLE_PEAK, ABOVE_SPAN):
    check(
        "embed",
        f"mock: {_case.name} still yields a unit vector",
        lambda c=_case: MOCK_EMBEDDERS["dreams"].embed_spectrum(
            c.peaks, precursor_mz=c.precursor_mz
        ),
        predicate=vector_ok,
        input_status=f"{len(_case.peaks)} peak(s), precursor {_case.precursor_mz}",
        assertion="shape (1024,), float32, L2 norm == 1 (bins clipped to the grid)",
    )

_zero_vec = MOCK_EMBEDDERS["dreams"].embed_spectrum(
    ZERO_INTENSITY.peaks, precursor_mz=ZERO_INTENSITY.precursor_mz
)
check(
    "embed",
    "mock: all-zero intensities produce a degenerate zero vector",
    lambda: _zero_vec,
    predicate=lambda v: expect(
        float(np.linalg.norm(v)) == 0.0,
        f"expected a zero-norm vector, got norm={float(np.linalg.norm(v))}",
    ),
    input_status="2 peaks, both intensity 0.0",
    assertion="documented degenerate case: norm 0 (so scoring must guard norm==0)",
)

# The tool layer must absorb that degenerate vector without producing NaN.
check(
    "compute_cosine",
    "dreams (mock) scores a zero-intensity spectrum as 0.0, not NaN",
    lambda: sync_call(
        "compute_cosine",
        query_peaks=ZERO_INTENSITY.peaks.tolist(),
        reference_peaks=CAFFEINE_MS2.peaks.tolist(),
        scoring_method="dreams",
    ),
    predicate=lambda text: (
        must_contain(
            text, "Cosine Similarity (DreaMS): **0.0000**", "mock (dev/test-only"
        ),
        expect("nan" not in text.lower(), "NaN leaked into the report"),
    )[-1],
    input_status="zero-intensity query vs caffeine reference",
    assertion="score 0.0000, no NaN, mock backend label present in the report",
)

# A noisy spectrum must stay close to its clean counterpart and remain
# distinguishable from an unrelated compound.
_noisy_self = float(
    np.dot(
        MOCK_EMBEDDERS["dreams"].embed_spectrum(
            NOISY_CAFFEINE.peaks, precursor_mz=NOISY_CAFFEINE.precursor_mz
        ),
        MOCK_EMBEDDERS["dreams"].embed_spectrum(
            CAFFEINE_MS2.peaks, precursor_mz=CAFFEINE_MS2.precursor_mz
        ),
    )
)
_noisy_cross = float(
    np.dot(
        MOCK_EMBEDDERS["dreams"].embed_spectrum(
            NOISY_CAFFEINE.peaks, precursor_mz=NOISY_CAFFEINE.precursor_mz
        ),
        MOCK_EMBEDDERS["dreams"].embed_spectrum(
            GLUCOSE_MS2.peaks, precursor_mz=GLUCOSE_MS2.precursor_mz
        ),
    )
)
check(
    "embed",
    "noise robustness: noisy caffeine is nearer caffeine than glucose",
    lambda: (_noisy_self, _noisy_cross),
    predicate=lambda pair: expect(
        pair[0] > 0.90 and pair[0] > pair[1] + 0.1,
        f"ranking/robustness failed: self={pair[0]:.4f}, cross={pair[1]:.4f}",
    ),
    input_status="caffeine + 3% baseline noise",
    assertion=f"self-cosine {_noisy_self:.4f} > 0.90 and > glucose cosine {_noisy_cross:.4f} + 0.1",
)

In [ ]:
# =============================================================================
# 3.4 DreaMSInferenceEmbedder — real adapter, availability & input guards
# =============================================================================
DREAMS_INSTALLED = importlib.util.find_spec("dreams") is not None
TORCH_INSTALLED = importlib.util.find_spec("torch") is not None

check(
    "DreaMSInferenceEmbedder",
    "check_available() reports the true install state",
    DreaMSInferenceEmbedder.check_available,
    raises=None if DREAMS_INSTALLED else EmbeddingBackendUnavailable,
    predicate=None
    if DREAMS_INSTALLED
    else (lambda exc: must_contain(str(exc), "dreams", "install")),
    input_status=f"dreams installed={DREAMS_INSTALLED}, torch installed={TORCH_INSTALLED}",
    assertion=(
        "adapter instantiable (real inference available)"
        if DREAMS_INSTALLED
        else "EmbeddingBackendUnavailable raised with install instructions"
    ),
)

# Input guards run before the model is touched, so they stay deterministic even
# without the `dreams` package (and never trigger a checkpoint download).
check(
    "DreaMSInferenceEmbedder",
    "empty peak list is rejected before any model load",
    lambda: DreaMSInferenceEmbedder().embed_spectrum(EMPTY.peaks, precursor_mz=150.0),
    raises=ValueError,
    predicate=lambda exc: must_contain(str(exc), "non-empty (N, 2)"),
    input_status="shape (0, 2) array",
    assertion="ValueError from peak coercion (no checkpoint download attempted)",
)

check(
    "DreaMSInferenceEmbedder",
    "no fragments below the precursor is rejected (DataFormat-A)",
    lambda: DreaMSInferenceEmbedder().embed_spectrum(
        ABOVE_PRECURSOR.peaks, precursor_mz=ABOVE_PRECURSOR.precursor_mz
    ),
    raises=ValueError,
    predicate=lambda exc: must_contain(
        str(exc), "no fragment peaks below the precursor"
    ),
    input_status="2 peaks at/above precursor 100.0",
    assertion="ValueError explaining the DataFormat-A fragment rule",
)

if DREAMS_INSTALLED and ALLOW_REAL_INFERENCE:
    check(
        "DreaMSInferenceEmbedder",
        "real 1024-d embedding of the caffeine spectrum",
        lambda: DreaMSInferenceEmbedder().embed_spectrum(
            CAFFEINE_MS2.peaks, precursor_mz=CAFFEINE_MS2.precursor_mz
        ),
        predicate=vector_ok,
        input_status="9 caffeine fragments, precursor 194.0804",
        assertion="real inference returns shape (1024,), float32, L2 norm == 1",
    )
else:
    skip(
        "DreaMSInferenceEmbedder",
        "real 1024-d embedding of the caffeine spectrum",
        "requires the optional `dreams` package"
        if not DREAMS_INSTALLED
        else "set MSMCP_EVAL_ALLOW_REAL_INFERENCE=1 to run the checkpoint",
        input_status=f"dreams installed={DREAMS_INSTALLED}",
    )

check(
    "LSMMS2InferenceEmbedder",
    "check_available() requires a configured checkpoint",
    LSMMS2InferenceEmbedder.check_available,
    raises=None
    if os.environ.get("MSMCP_LSM_MS2_CKPT")
    else EmbeddingBackendUnavailable,
    predicate=(
        None
        if os.environ.get("MSMCP_LSM_MS2_CKPT")
        else (lambda exc: must_contain(str(exc), "MSMCP_LSM_MS2_CKPT"))
    ),
    input_status=f"MSMCP_LSM_MS2_CKPT={os.environ.get('MSMCP_LSM_MS2_CKPT', '<unset>')}",
    assertion="EmbeddingBackendUnavailable naming the checkpoint environment variable",
)

# get_embedder() resolution: the explicit mock flag, and strict rejection.
check(
    "get_embedder",
    "backend='mock' resolves to the deterministic adapter",
    lambda: type(get_embedder("dreams", backend="mock")).__name__,
    predicate=lambda name: expect(name == "DreaMSEmbedder", f"resolved {name}"),
    input_status="backend='mock'",
    assertion="resolves to DreaMSEmbedder (explicit dev/test flag honoured)",
)

check(
    "get_embedder",
    "unknown embedding method is rejected",
    lambda: get_embedder("orbitrap-ai", backend="mock"),
    raises=ValueError,
    predicate=lambda exc: must_contain(str(exc), "Unknown embedding method", "lsm-ms2"),
    input_status="method='orbitrap-ai'",
    assertion="ValueError listing the registered methods",
)

In [ ]:
# =============================================================================
# 3.5 Tool-level scoring paths (classical / dreams / lsm-ms2)
# =============================================================================
check(
    "compute_cosine",
    "classical scoring of identical spectra == 1.0",
    lambda: sync_call(
        "compute_cosine",
        query_peaks=CAFFEINE_MS2.peaks.tolist(),
        reference_peaks=CAFFEINE_MS2.peaks.tolist(),
    ),
    predicate=lambda text: must_contain(
        text,
        "Cosine Similarity: **1.0000**",
        "Matched: 9 / 9 query peaks (100.0%)",
        "Reference peaks utilised: 9 / 9",
    ),
    input_status="9 identical peaks on both sides, tolerance 0.02 Da",
    assertion="score 1.0000 and full matched-peak accounting",
)

check(
    "compute_cosine",
    "classical scoring tolerates sub-tolerance m/z jitter",
    lambda: sync_call(
        "compute_cosine",
        query_peaks=CAFFEINE_MS2.peaks.tolist(),
        reference_peaks=(CAFFEINE_MS2.peaks + np.array([0.008, 0.0])).tolist(),
        ms2_tolerance=0.02,
    ),
    predicate=lambda text: must_contain(text, "Cosine Similarity: **1.0000**"),
    input_status="+0.008 Da shift (< 0.02 Da tolerance)",
    assertion="greedy matching pairs the shifted peaks; score stays 1.0000",
)

for _method, _label in (("dreams", "DreaMS"), ("lsm-ms2", "LSM-MS2")):
    check(
        "compute_cosine",
        f"{_method} (mock) embedding score is labelled as non-learned",
        lambda m=_method: sync_call(
            "compute_cosine",
            query_peaks=CAFFEINE_MS2.peaks.tolist(),
            reference_peaks=GLUCOSE_MS2.peaks.tolist(),
            scoring_method=m,
        ),
        predicate=lambda text, lbl=_label: must_contain(
            text,
            f"({lbl})",
            f"{EMBEDDING_DIM}-d",
            "mock (dev/test-only, not a learned model)",
        ),
        input_status="caffeine query vs glucose reference",
        assertion="report names the model, the 1024-d space and the mock/dev-only backend",
    )

check(
    "compute_cosine",
    "out-of-range tolerance is rejected by the schema",
    lambda: sync_call(
        "compute_cosine",
        query_peaks=CAFFEINE_MS2.peaks.tolist(),
        reference_peaks=GLUCOSE_MS2.peaks.tolist(),
        ms2_tolerance=2.5,
    ),
    raises=ValidationError,
    input_status="ms2_tolerance=2.5 (> 1.0 maximum)",
    assertion="Pydantic le=1.0 bound enforced (ValidationError)",
)

check(
    "compute_cosine",
    "negative intensity is reported as an ERROR string",
    lambda: sync_call(
        "compute_cosine",
        query_peaks=[[100.0, -1.0]],
        reference_peaks=[[100.0, 1.0]],
    ),
    predicate=lambda result: must_contain(result, "ERROR", "negative intensity"),
    input_status="query peak [100.0, -1.0]",
    assertion="ERROR string at the tool boundary (not a traceback)",
)


# Real-mode guard: with MSMCP_EMBEDDING_BACKEND=real the tool must refuse to
# invent a score rather than silently falling back to a mock.
def real_mode_probe() -> str:
    previous = os.environ.get("MSMCP_EMBEDDING_BACKEND")
    os.environ["MSMCP_EMBEDDING_BACKEND"] = "real"
    try:
        return sync_call(
            "compute_cosine",
            query_peaks=CAFFEINE_MS2.peaks.tolist(),
            reference_peaks=GLUCOSE_MS2.peaks.tolist(),
            scoring_method="dreams",
        )
    finally:
        os.environ["MSMCP_EMBEDDING_BACKEND"] = previous or "mock"


if not DREAMS_INSTALLED and not ALLOW_REAL_INFERENCE:
    check(
        "compute_cosine",
        "production mode refuses to fabricate an embedding score",
        real_mode_probe,
        raises=EmbeddingBackendUnavailable,
        predicate=lambda exc: must_contain(str(exc), "not installed"),
        input_status="backend forced to 'real', dreams absent",
        assertion="EmbeddingBackendUnavailable propagates (no silent mock fallback)",
    )
else:
    skip(
        "compute_cosine",
        "production mode refuses to fabricate an embedding score",
        "real inference is installed/allowed in this environment",
        input_status=f"dreams installed={DREAMS_INSTALLED}",
    )


def bench_mock_embedding(iterations: int = 250) -> float:
    """Mean wall-clock ms for one 1024-d mock embedding."""
    started = time.perf_counter()
    for _ in range(iterations):
        MOCK_EMBEDDERS["dreams"].embed_spectrum(
            CAFFEINE_MS2.peaks, precursor_mz=CAFFEINE_MS2.precursor_mz
        )
    return (time.perf_counter() - started) * 1000.0 / iterations


check(
    "embed",
    "mock embedding latency (runtime_ms = mean per spectrum)",
    bench_mock_embedding,
    predicate=lambda ms: expect(ms < 50.0, f"unexpectedly slow: {ms:.3f} ms/spectrum"),
    input_status=f"{len(CAFFEINE_MS2.peaks)}-peak spectrum x250",
    assertion="mean latency < 50 ms per 1024-d embedding",
)

_s3_tools = {
    "embed",
    "compute_cosine",
    "get_embedder",
    "DreaMSInferenceEmbedder",
    "LSMMS2InferenceEmbedder",
}
_s3_rows = [r for r in RESULTS if r.tool in _s3_tools]
print(
    f"\nSection 3: {len(_s3_rows)} cases | "
    f"PASS {sum(r.status == 'PASS' for r in _s3_rows)} | "
    f"FAIL {sum(r.status == 'FAIL' for r in _s3_rows)} | "
    f"SKIP {sum(r.status == 'SKIP' for r in _s3_rows)}"
)

## 4. Async Library Search & Job Polling

`search_library` submits the scan to a `JobExecutor` and returns a job ID
immediately; `check_search_status` is the poller. This section drives the
full state machine `queued → running → completed | failed | cancelled` and
cross-checks every MCP-visible payload against the executor's own state
(white-box introspection of `msmcp.tools.search._EXECUTOR`).

It starts with the **real** file-backed tools (`load_mzml_summary`,
`generate_qc_summary`) run against mzML fixtures this notebook writes, which
also verifies the filesystem security boundary.

The search is half real and half synthetic, and the notebook asserts exactly
that: the **query** spectrum is read for real (and a missing or malformed one
is refused before dispatch), while the **library** file is still never opened
because MSMCP has no spectral-library reader. Reports carry a banner saying
so.

Covered: successful scans (classical and `dreams` mock), real query reading
and its failure modes, unknown / lost jobs, cooperative cancellation, and the
concurrency cap (`_EXECUTOR.max_concurrency`).

In [ ]:
# =============================================================================
# 4.1 Sample acquisition files (real mzML: the query is read, the library is not)
# =============================================================================
_MZML_NS = "http://psi.hupo.org/ms/mzml"


def _binary_array(kind: str, values: list[float]) -> str:
    accession, name = (
        ("MS:1000514", "m/z array")
        if kind == "mz"
        else ("MS:1000515", "intensity array")
    )
    payload = base64.b64encode(np.asarray(values, dtype="<f8").tobytes()).decode()
    return (
        f'<binaryDataArray encodedLength="{len(payload)}">'
        f'<cvParam cvRef="MS" accession="{accession}" name="{name}" value=""/>'
        f'<cvParam cvRef="MS" accession="MS:1000523" name="64-bit float" value=""/>'
        f'<cvParam cvRef="MS" accession="MS:1000576" name="no compression" value=""/>'
        f"<binary>{payload}</binary>"
        f"</binaryDataArray>"
    )


def _spectrum_xml(
    index: int, ms_level: int, precursor: float | None, pairs: list[tuple[float, float]]
) -> str:
    mz = [p[0] for p in pairs]
    intensity = [p[1] for p in pairs]
    precursor_xml = ""
    if precursor is not None:
        precursor_xml = (
            f'<precursorList count="1"><precursor><selectedIonList count="1"><selectedIon>'
            f'<cvParam cvRef="MS" accession="MS:1000744" name="selected ion m/z" value="{precursor}"/>'
            f"</selectedIon></selectedIonList></precursor></precursorList>"
        )
    return (
        f'<spectrum index="{index}" id="scan={index}" defaultArrayLength="{len(pairs)}">'
        f'<cvParam cvRef="MS" accession="MS:1000511" name="ms level" value="{ms_level}"/>'
        f'<scanList count="1"><scan><cvParam cvRef="MS" accession="MS:1000016" '
        f'name="scan start time" value="{float(index)}" unitAccession="MS:1000038" unitName="second"/>'
        f"</scan></scanList>"
        f"{precursor_xml}"
        f'<binaryDataArrayList count="2">'
        f"{_binary_array('mz', mz)}{_binary_array('intensity', intensity)}"
        f"</binaryDataArrayList>"
        f"</spectrum>"
    )


def write_mzml(
    path: Path, spectra: list[tuple[int, float | None, list[tuple[float, float]]]]
) -> Path:
    body = "".join(
        _spectrum_xml(i, level, precursor, pairs)
        for i, (level, precursor, pairs) in enumerate(spectra)
    )
    path.write_text(
        f'<?xml version="1.0" encoding="utf-8"?>'
        f'<mzML xmlns="{_MZML_NS}" version="1.1.0"><run id="eval">'
        f'<spectrumList count="{len(spectra)}" defaultDataProcessingRef="dp">{body}</spectrumList>'
        f"</run></mzML>",
        encoding="utf-8",
    )
    return path


EXPERIMENTAL_MZML = write_mzml(
    ARTIFACT_DIR / "caffeine_experimental.mzML",
    [
        (2, CAFFEINE_MS2.precursor_mz, [tuple(p) for p in CAFFEINE_MS2.peaks]),
        (2, GLUCOSE_MS2.precursor_mz, [tuple(p) for p in GLUCOSE_MS2.peaks]),
        (1, None, [(120.0, 1.0e5), (250.0, 4.0e4)]),
    ],
)
LIBRARY_MZML = write_mzml(
    ARTIFACT_DIR / "reference_library.mzML",
    [
        (2, 194.0804, [(138.0662, 4.2e5), (110.0713, 2.1e5)]),
        (2, 180.0634, [(163.0601, 5.1e5)]),
    ],
)
TRUNCATED_MZML = ARTIFACT_DIR / "truncated.mzML"
TRUNCATED_MZML.write_text(
    EXPERIMENTAL_MZML.read_text(encoding="utf-8")[:400], encoding="utf-8"
)
MISSING_MZML = ARTIFACT_DIR / "does_not_exist.mzML"

for _path in (EXPERIMENTAL_MZML, LIBRARY_MZML, TRUNCATED_MZML):
    print(f"{_path.name:<30} {_path.stat().st_size:>8,} bytes")
print(f"{MISSING_MZML.name:<30} (never created — proves the query file is validated)")

In [ ]:
# =============================================================================
# 4.2 Real file-backed tools — mzML parsing, QC and the security boundary
# =============================================================================
check(
    "load_mzml_summary",
    "summarises the 3-spectrum fixture it was given",
    lambda: sync_call("load_mzml_summary", file_path=str(EXPERIMENTAL_MZML)),
    predicate=lambda text: must_contain(
        text,
        f"File: {EXPERIMENTAL_MZML.name}",
        "Format: MZML",
        "Summarised 3 of 3 total spectra.",
    ),
    input_status=f"{EXPERIMENTAL_MZML.name} (3 spectra, 2x MS2 + 1x MS1)",
    assertion="real XML parse; declared spectrum count matches the fixture",
)

check(
    "load_mzml_summary",
    "max_spectra caps the summary but reports the true total",
    lambda: sync_call(
        "load_mzml_summary", file_path=str(EXPERIMENTAL_MZML), max_spectra=1
    ),
    predicate=lambda text: (
        must_contain(text, "Summarised 1 of 3 total spectra.", "raise `max_spectra`"),
    )[-1],
    input_status="max_spectra=1 on the 3-spectrum fixture",
    assertion="truncated listing with an explicit 'of 3 total' footer",
)

check(
    "generate_qc_summary",
    "QC report covers SNR, peak density and diagnostic ions",
    lambda: sync_call("generate_qc_summary", file_path=str(EXPERIMENTAL_MZML)),
    predicate=lambda text: must_contain(
        text,
        "## QC Summary Report",
        "**Spectra analysed:** 3",
        "### Signal-to-Noise Ratio (estimated)",
        "### Peak Density",
        "### Diagnostic Fragment Analysis",
    ),
    input_status=f"{EXPERIMENTAL_MZML.name} (3 spectra)",
    assertion="all four report sections present; 3 spectra analysed",
)

check(
    "load_mzml_summary",
    "a truncated mzML is reported as malformed, not silently parsed",
    lambda: sync_call("load_mzml_summary", file_path=str(TRUNCATED_MZML)),
    raises=MalformedFileError,
    input_status=f"{TRUNCATED_MZML.name} (400 bytes of a cut-off document)",
    assertion="MalformedFileError from the real parser",
)

_escape_probe = Path(tempfile.gettempdir()) / "msmcp_eval_outside_root.mzML"
_escape_probe.write_text(
    "this file sits outside the server's allowed root", encoding="utf-8"
)
if DEFAULT_POLICY.allowed_root != Path("/"):
    check(
        "load_mzml_summary",
        "a path outside the allowed root is refused (security boundary)",
        lambda: sync_call("load_mzml_summary", file_path=str(_escape_probe)),
        raises=PathEscapeError,
        predicate=lambda exc: must_contain(str(exc), "outside the allowed root"),
        input_status=f"{_escape_probe} vs root {DEFAULT_POLICY.allowed_root}",
        assertion="PathEscapeError before any file access",
    )
else:
    skip(
        "load_mzml_summary",
        "a path outside the allowed root is refused (security boundary)",
        "MSMCP_ALLOWED_ROOT is '/' so every path is inside the boundary",
        input_status=f"allowed root {DEFAULT_POLICY.allowed_root}",
    )

In [ ]:
# =============================================================================
# 4.3 Dispatch / poll helpers
# =============================================================================
JOB_ID_RE = re.compile(r"Job ID:\*\* `([0-9a-f]{32})`")
_TERMINAL = ("completed", "failed", "cancelled")
_STATE_ORDER = {"pending": 0, "running": 1, "completed": 2, "failed": 2, "cancelled": 2}
JOB_STARTED: dict[str, float] = {}
JOB_DISPATCH_MS: dict[str, float] = {}


def executor_kind(job_id: str) -> str:
    """Job state sampled from the executor, in the poller's vocabulary.

    ``check_search_status`` renders the ``queued``/``running`` phases as
    "Pending"/"Running"; the executor exposes the MCP-aligned status
    ("working") plus a finer phase, so map it back for comparison.
    """
    try:
        snapshot = _EXECUTOR.status(job_id)
    except UnknownJobError:
        return "missing"
    if snapshot.status == "working":
        return "pending" if snapshot.phase == "queued" else "running"
    return snapshot.status


def payload_kind(text: str) -> str:
    """Classify an MCP-visible poller payload into a job status."""
    if "⏳ **Pending**" in text:
        return "pending"
    if "🔄 **Running**" in text:
        return "running"
    if "❌ **Failed" in text:
        return "failed"
    if "🚫 **Cancelled**" in text:
        return "cancelled"
    # A finished report is delivered once; later polls answer with a digest, and
    # both are "completed" payloads.  The digest is matched by its marker, not by
    # absence of the report heading, so a payload that is neither is still
    # reported as "unknown".
    if "✅ **Completed**" in text:
        return "completed"
    if "## Spectral Library Search Results" in text:
        return "completed"
    return "unknown"


def dispatch(
    experimental: Path | str,
    database: Path | str,
    method: str = "classical",
    chunk_size: int = 2000,
) -> tuple[str, str]:
    """Call search_library and return (job_id, raw dispatch payload).

    The dispatch latency is recorded per job so the end-to-end figure can be
    split into 'time to return a job id' and 'time to scan'.
    """
    started = time.perf_counter()
    text = async_call(
        "search_library",
        experimental_file=str(experimental),
        database_file=str(database),
        scoring_method=method,
        chunk_size=chunk_size,
    )
    elapsed_ms = (time.perf_counter() - started) * 1000.0
    match = JOB_ID_RE.search(text)
    if match is None:
        raise AssertionError(f"no job id in dispatch payload:\n{text}")
    job_id = match.group(1)
    uuid.UUID(job_id)  # the 32-hex UUID shape is part of the contract
    JOB_STARTED[job_id] = started
    JOB_DISPATCH_MS[job_id] = elapsed_ms
    return job_id, text


@dataclass
class PollOutcome:
    job_id: str
    status: str
    transitions: list[str]
    payload_kinds: list[str]
    mismatches: list[str]
    polls: int
    poll_ms: float
    job_ms: float
    dispatch_ms: float
    final_payload: str


def poll_job(
    job_id: str, *, timeout_s: float = 240.0, interval_s: float = 0.15
) -> PollOutcome:
    """Poll check_search_status until the job reaches a terminal state.

    Every payload is classified and compared with the executor's own state,
    so a human-readable status that disagrees with the machine state is
    caught.
    """
    deadline = time.perf_counter() + timeout_s
    started = time.perf_counter()
    transitions: list[str] = []
    kinds: list[str] = []
    mismatches: list[str] = []
    polls = 0
    final_payload = ""

    while time.perf_counter() < deadline:
        final_payload = async_call("check_search_status", job_id=job_id)
        polls += 1
        store_status = executor_kind(job_id)
        kind = payload_kind(final_payload)
        if not transitions or transitions[-1] != store_status:
            transitions.append(store_status)
        if not kinds or kinds[-1] != kind:
            kinds.append(kind)
        # The executor may advance between the payload being rendered and its
        # state being sampled (the scan runs on a worker thread), so a payload
        # that lags it is benign. A payload that is *ahead*, or a terminal
        # disagreement, is a real inconsistency.
        kind_rank = _STATE_ORDER.get(kind, -1)
        store_rank = _STATE_ORDER.get(store_status, -1)
        benign_lag = kind in ("pending", "running") and store_rank > kind_rank
        if kind != store_status and not benign_lag:
            mismatches.append(f"poll {polls}: payload={kind!r} store={store_status!r}")
        if store_status in _TERMINAL:
            # The payload read above may have been rendered a moment *before*
            # the job settled (the worker thread runs concurrently with this
            # poll), so re-read it now that the state is terminal.  Without
            # this, a job that fails or completes between rendering and
            # sampling returns a stale "Pending"/"Running" payload.
            # Re-read with full_report=True: the plain read above may already
            # have *delivered* the report, in which case a second plain read
            # would return the digest instead of the report this helper's
            # callers assert on.
            final_payload = async_call(
                "check_search_status", job_id=job_id, full_report=True
            )
            polls += 1
            terminal_kind = payload_kind(final_payload)
            if kinds[-1] != terminal_kind:
                kinds.append(terminal_kind)
            if terminal_kind != store_status:
                mismatches.append(
                    f"final poll {polls}: payload={terminal_kind!r} "
                    f"store={store_status!r}"
                )
            now = time.perf_counter()
            return PollOutcome(
                job_id,
                store_status,
                transitions,
                kinds,
                mismatches,
                polls,
                (now - started) * 1000.0,
                (now - JOB_STARTED.get(job_id, started)) * 1000.0,
                JOB_DISPATCH_MS.get(job_id, float("nan")),
                final_payload,
            )
        time.sleep(interval_s)

    raise TimeoutError(
        f"job {job_id} still {transitions[-1]!r} after {timeout_s:.0f}s "
        f"(polls={polls}, kinds={kinds})"
    )


def run_search(
    experimental: Path | str,
    database: Path | str,
    method: str = "classical",
    chunk_size: int = 2000,
) -> tuple[str, str, PollOutcome]:
    """Dispatch a search and poll it to a terminal state (one timed unit)."""
    job_id, payload = dispatch(experimental, database, method, chunk_size)
    return job_id, payload, poll_job(job_id)


def lifecycle_ok(outcome: PollOutcome) -> None:
    """The completed-state contract: monotone transitions, agreeing payloads."""
    expect(outcome.status == "completed", f"terminal status {outcome.status}")
    expect(outcome.mismatches == [], f"poller/store disagreement: {outcome.mismatches}")
    # The first observed state is normally queued or running.  On a very fast
    # machine a small scan can settle before the first poll, which is legal:
    # the job did complete.  What must never happen is a non-monotone history.
    expect(
        outcome.transitions[0] in ("pending", "running", "completed"),
        f"start state {outcome.transitions}",
    )
    if outcome.transitions[0] == "completed":
        expect(
            len(outcome.transitions) == 1,
            f"a job that started completed cannot move: {outcome.transitions}",
        )
    expect(outcome.transitions[-1] == "completed", f"end state {outcome.transitions}")
    orders = [_STATE_ORDER[s] for s in outcome.transitions]
    expect(orders == sorted(orders), f"non-monotone lifecycle: {outcome.transitions}")
    expect(
        all(k in ("pending", "running", "completed") for k in outcome.payload_kinds),
        f"unexpected payload kinds: {outcome.payload_kinds}",
    )


def completed_report(*needles: str) -> Callable[[tuple], None]:
    """Predicate factory: the job completed and its report contains *needles*."""

    def predicate(triple: tuple[str, str, PollOutcome]) -> None:
        _job_id, _payload, outcome = triple
        lifecycle_ok(outcome)
        must_contain(outcome.final_payload, *needles)

    return predicate


print("dispatch/poll helpers ready")

In [ ]:
# =============================================================================
# 4.4 Happy path — a classical scan, timed end to end
# =============================================================================
SEARCH_TRIPLE = check(
    "search_library",
    "end-to-end round trip: dispatch → running → report",
    lambda: run_search(EXPERIMENTAL_MZML, LIBRARY_MZML, "classical", 500),
    predicate=completed_report(
        "SYNTHETIC LIBRARY — NOT A COMPOUND IDENTIFICATION",
        "## Spectral Library Search Results",
        "Query spectrum:",
        "classical (greedy peak matching, ±0.02 Da)",
    ),
    input_status=f"{EXPERIMENTAL_MZML.name} (query, read) + {LIBRARY_MZML.name} (not read)",
    assertion="completed; report well-formed and honestly labelled (runtime_ms = dispatch + scan)",
)
JOB_CLASSICAL, DISPATCH_PAYLOAD, OUTCOME_CLASSICAL = SEARCH_TRIPLE

check(
    "search_library",
    "dispatch returns a job id without waiting for the scan",
    lambda: DISPATCH_PAYLOAD,
    predicate=lambda text: (
        must_contain(text, "## Search Dispatched", "synthetic", "check_search_status"),
        # Dispatch now reads and validates the real query spectrum before
        # submitting, so this compares against the scan rather than an absolute
        # budget: the point is that dispatch does not wait for the scan, and it
        # must stay well inside a host's tool-call timeout.
        expect(
            OUTCOME_CLASSICAL.dispatch_ms < OUTCOME_CLASSICAL.job_ms,
            f"dispatch ({OUTCOME_CLASSICAL.dispatch_ms:.0f} ms) did not beat the "
            f"scan ({OUTCOME_CLASSICAL.job_ms:.0f} ms)",
        ),
        expect(
            OUTCOME_CLASSICAL.dispatch_ms < 2000.0,
            f"dispatch blocked for {OUTCOME_CLASSICAL.dispatch_ms:.0f} ms",
        ),
    )[-1],
    input_status=f"job {JOB_CLASSICAL[:8]}… (already completed)",
    assertion=(
        f"returned in {OUTCOME_CLASSICAL.dispatch_ms:.1f} ms while the scan took "
        f"≈{OUTCOME_CLASSICAL.job_ms:.0f} ms"
    ),
)

check(
    "check_search_status",
    "lifecycle payloads agree with the executor state",
    lambda: OUTCOME_CLASSICAL,
    predicate=lifecycle_ok,
    input_status=f"job {JOB_CLASSICAL[:8]}…",
    assertion=(
        f"polls={OUTCOME_CLASSICAL.polls}, transitions="
        f"{'→'.join(OUTCOME_CLASSICAL.transitions)}, "
        f"payload kinds={'→'.join(OUTCOME_CLASSICAL.payload_kinds)}, "
        f"scan ≈{OUTCOME_CLASSICAL.job_ms:.0f} ms"
    ),
)

check(
    "check_search_status",
    "completed payload is the full, honestly-labelled report",
    lambda: OUTCOME_CLASSICAL.final_payload,
    predicate=lambda text: (
        must_contain(
            text,
            "not opened",
            "| Rank | Compound"
            if "| Rank | Compound" in text
            else "No hits passed the significance threshold",
            "Query spectrum:",
        ),
        expect(
            text.count("not opened") >= 1,
            f"library path echoed as 'not opened' {text.count('not opened')}x",
        ),
        expect(
            text.index("SYNTHETIC LIBRARY")
            < text.index(
                "| Rank | Compound"
                if "| Rank | Compound" in text
                else "No hits passed the significance threshold"
            ),
            "provenance banner must precede the results block",
        ),
        expect(
            "Provenance: search_library" in text,
            "the report must carry a structured provenance footer",
        ),
    )[-1],
    input_status="final poll payload",
    assertion="banner first; results block present (hit table or no-hit line); provenance footer present",
)

print("\n--- report head -------------------------------------------------")
print("\n".join(OUTCOME_CLASSICAL.final_payload.splitlines()[:8]))

In [ ]:
# =============================================================================
# 4.5 Variants — chunking, embedding scorer, unread & malformed inputs
# =============================================================================
check(
    "search_library",
    "chunk_size=100 (documented minimum) renders the same report",
    lambda: run_search(EXPERIMENTAL_MZML, LIBRARY_MZML, "classical", 100),
    predicate=completed_report("## Spectral Library Search Results"),
    input_status="chunk_size=100",
    assertion="chunked scan completes and renders the same report skeleton",
)

check(
    "search_library",
    "dreams (mock) scorer path completes and is labelled",
    lambda: run_search(EXPERIMENTAL_MZML, LIBRARY_MZML, "dreams", 2000),
    predicate=completed_report("DreaMS", "## Spectral Library Search Results"),
    input_status="scoring_method='dreams' (mock backend)",
    assertion="job completes through the embedding-scorer path",
)

check(
    "search_library",
    "a nonexistent query file is refused before the job is dispatched",
    lambda: async_call(
        "search_library",
        experimental_file=str(MISSING_MZML),
        database_file=str(LIBRARY_MZML),
    ),
    raises=InaccessiblePathError,
    input_status=f"{MISSING_MZML.name} (never created on disk)",
    assertion="InaccessiblePathError — the query is genuinely read, and fails loudly",
)

check(
    "search_library",
    "a malformed (truncated) query is reported, not silently searched",
    lambda: async_call(
        "search_library",
        experimental_file=str(TRUNCATED_MZML),
        database_file=str(LIBRARY_MZML),
    ),
    raises=MalformedFileError,
    input_status=f"{TRUNCATED_MZML.name} (400 bytes of a cut-off document)",
    assertion="MalformedFileError from the real parser, raised before dispatch",
)

check(
    "search_library",
    "the library path is still not opened (only the query is read)",
    lambda: run_search(EXPERIMENTAL_MZML, MISSING_MZML, "classical"),
    predicate=completed_report(
        "## Spectral Library Search Results", f"`{MISSING_MZML}` — not opened"
    ),
    input_status=f"query={EXPERIMENTAL_MZML.name} (real), library={MISSING_MZML.name} (absent)",
    assertion="job completes and the report states the library was not opened",
)

In [ ]:
# =============================================================================
# 4.6 Negative paths — schema guards, unknown jobs, lost/orphaned jobs
# =============================================================================
check(
    "search_library",
    "chunk_size below the documented minimum is rejected",
    lambda: async_call(
        "search_library",
        experimental_file="a.mzML",
        database_file="b.mzML",
        chunk_size=50,
    ),
    raises=ValidationError,
    input_status="chunk_size=50 (< ge=100)",
    assertion="Pydantic ge=100 bound enforced (ValidationError)",
)

check(
    "search_library",
    "unknown scoring_method is rejected",
    lambda: async_call(
        "search_library",
        experimental_file="a.mzML",
        database_file="b.mzML",
        scoring_method="banana",
    ),
    raises=ValidationError,
    input_status="scoring_method='banana'",
    assertion="Literal['classical','dreams','lsm-ms2'] enforced (ValidationError)",
)

check(
    "check_search_status",
    "malformed job id is reported, not treated as pending",
    lambda: async_call("check_search_status", job_id="not-a-uuid"),
    predicate=lambda text: must_contain(text, "Unknown Job", "not a valid job ID"),
    input_status="job_id='not-a-uuid'",
    assertion="explicit Unknown Job payload (never a hang)",
)

LOST_JOB_ID = uuid.uuid4().hex
check(
    "check_search_status",
    "well-formed but unknown job id reports failure",
    lambda: async_call("check_search_status", job_id=LOST_JOB_ID),
    predicate=lambda text: must_contain(
        text, "Failed (not found)", "does **not** survive a server restart"
    ),
    input_status=f"random UUID {LOST_JOB_ID[:8]}…",
    assertion="missing jobs are reported failed, never pending",
)

check(
    "cancel_search",
    "cancelling an unknown job reports unknown",
    lambda: async_call("cancel_search", job_id=LOST_JOB_ID),
    predicate=lambda text: must_contain(text, "Unknown Job"),
    input_status=f"random UUID {LOST_JOB_ID[:8]}…",
    assertion="Unknown Job payload; no state change",
)

check(
    "search_library",
    "a spectrum reference and a file path together are rejected",
    lambda: async_call(
        "search_library",
        experimental_file=str(EXPERIMENTAL_MZML),
        spectrum_reference="ptr:spectrum:deadbeef",
        database_file=str(LIBRARY_MZML),
    ),
    raises=ValidationError,
    input_status="both query sources supplied",
    assertion="exactly-one-source validator enforced (ValidationError)",
)

check(
    "search_library",
    "an unknown spectrum reference is refused before dispatch",
    lambda: async_call(
        "search_library",
        spectrum_reference=f"ptr:spectrum:{uuid.uuid4().hex}",
        database_file=str(LIBRARY_MZML),
    ),
    raises=MsmcpError,
    predicate=lambda exc: must_contain(str(exc), "No live data reference"),
    input_status="well-formed but unknown reference",
    assertion="UnknownReferenceError — a stale reference never becomes a silent empty query",
)

# A job the executor no longer knows about (the restart case) must be reported
# as failed rather than polled forever.
FORGOTTEN_ID, _ = dispatch(EXPERIMENTAL_MZML, LIBRARY_MZML, "classical", 5000)
_EXECUTOR.forget(FORGOTTEN_ID)
_forgotten_payload = async_call("check_search_status", job_id=FORGOTTEN_ID)
check(
    "check_search_status",
    "a forgotten/restarted job is failed instead of polling forever",
    lambda: _forgotten_payload,
    predicate=lambda text: must_contain(
        text, "Failed (not found)", "does **not** survive a server restart"
    ),
    input_status=f"job {FORGOTTEN_ID[:8]}… dropped from the executor",
    assertion="terminal 'Failed (not found)' with a restart explanation",
)

In [ ]:
# =============================================================================
# 4.7 Failure path — a scorer that cannot be loaded must fail the job
# =============================================================================
def run_search_with_real_backend() -> tuple[str, str, PollOutcome]:
    """Force strict real inference for the duration of one search round trip."""
    previous = os.environ.get("MSMCP_EMBEDDING_BACKEND")
    os.environ["MSMCP_EMBEDDING_BACKEND"] = "real"
    try:
        return run_search(EXPERIMENTAL_MZML, LIBRARY_MZML, "dreams")
    finally:
        os.environ["MSMCP_EMBEDDING_BACKEND"] = previous or "mock"


def failed_search_predicate(triple: tuple[str, str, PollOutcome]) -> None:
    _job_id, _payload, outcome = triple
    expect(outcome.status == "failed", f"status={outcome.status}")
    expect(outcome.mismatches == [], f"poller/store disagreement: {outcome.mismatches}")
    must_contain(outcome.final_payload, "EmbeddingBackendUnavailable")


if not DREAMS_INSTALLED and not ALLOW_REAL_INFERENCE:
    FAILED_TRIPLE = check(
        "search_library",
        "unavailable real backend drives the job to failed",
        lambda: run_search_with_real_backend(),
        predicate=failed_search_predicate,
        input_status="backend forced to 'real', dreams absent",
        assertion="terminal state failed; poller reports the EmbeddingBackendUnavailable traceback",
    )
    print("\n--- failure payload tail ---------------------------------------")
    print("\n".join(FAILED_TRIPLE[2].final_payload.splitlines()[-6:]))
else:
    skip(
        "search_library",
        "unavailable real backend drives the job to failed",
        "requires the dreams package to be absent (installed or real runs allowed)",
        input_status=f"dreams installed={DREAMS_INSTALLED}",
    )

In [ ]:
# =============================================================================
# 4.8 Cancellation — cooperative cancel of a live job
# =============================================================================
def cancel_roundtrip() -> tuple[str, str, PollOutcome]:
    """Dispatch, cancel immediately, then poll to a terminal state."""
    job_id, _ = dispatch(EXPERIMENTAL_MZML, LIBRARY_MZML, "classical", 100)
    payload = async_call("cancel_search", job_id=job_id)
    return job_id, payload, poll_job(job_id)


def cancel_predicate(triple: tuple[str, str, PollOutcome]) -> None:
    job_id, payload, outcome = triple
    must_contain(payload, "Job ID", job_id)
    expect(
        "Search Cancelled" in payload or "Not Cancelled" in payload,
        f"unexpected cancel payload:\n{payload}",
    )
    expect(outcome.status in ("cancelled", "completed"), f"status={outcome.status}")
    expect(outcome.mismatches == [], f"poller/store disagreement: {outcome.mismatches}")
    expect(
        payload_kind(outcome.final_payload) == outcome.status,
        f"payload kind {payload_kind(outcome.final_payload)!r} != store {outcome.status!r}",
    )


CANCEL_TRIPLE = check(
    "cancel_search",
    "cancel is acknowledged and the job settles in a terminal state",
    cancel_roundtrip,
    predicate=cancel_predicate,
    input_status="job cancelled immediately after dispatch",
    assertion=(
        "cancel acknowledged; job reached a terminal state whose poller payload "
        "matches the store (cancellation is cooperative, so completion is legal)"
    ),
)
print(
    f"cancellation outcome: {CANCEL_TRIPLE[2].status} "
    f"(transitions={'→'.join(CANCEL_TRIPLE[2].transitions)})"
)

check(
    "cancel_search",
    "a completed job cannot be cancelled",
    lambda: async_call("cancel_search", job_id=JOB_CLASSICAL),
    predicate=lambda text: must_contain(text, "Not Cancelled", "already", "completed"),
    input_status=f"job {JOB_CLASSICAL[:8]}… (terminal)",
    assertion="terminal-state guard: no cancellation of finished jobs",
)

In [ ]:
# =============================================================================
# 4.9 Concurrency cap — five simultaneous searches, at most four on CPU
# =============================================================================
STRESS_COUNT = 5


def stress_run() -> tuple[list[str], int, int, bool, float]:
    """Dispatch STRESS_COUNT searches, watch the store, wait for completion."""
    started = time.perf_counter()
    job_ids = [
        dispatch(EXPERIMENTAL_MZML, LIBRARY_MZML, "classical", 100)[0]
        for _ in range(STRESS_COUNT)
    ]
    max_running = 0
    max_pending = 0
    pending_seen = False
    last_signature = None
    deadline = time.perf_counter() + 300.0
    while time.perf_counter() < deadline:
        states = [executor_kind(jid) for jid in job_ids]
        max_running = max(max_running, states.count("running"))
        max_pending = max(max_pending, states.count("pending"))
        pending_seen = pending_seen or "pending" in states
        signature = tuple(sorted(states))
        if signature != last_signature:
            print(f"  t+{time.perf_counter() - SUITE_START:6.2f}s  {signature}")
            last_signature = signature
        if all(state in _TERMINAL for state in states):
            break
        time.sleep(0.05)

    final_states = [executor_kind(jid) for jid in job_ids]
    for jid in job_ids:
        _EXECUTOR.forget(jid)
    return (
        final_states,
        max_running,
        max_pending,
        pending_seen,
        (time.perf_counter() - started) * 1000.0,
    )


STRESS_RESULT = check(
    "search_library",
    f"{STRESS_COUNT} concurrent searches respect the concurrency cap",
    stress_run,
    predicate=lambda data: (
        expect(
            all(state == "completed" for state in data[0]),
            f"not all jobs completed: {data[0]}",
        ),
        expect(
            data[1] <= _EXECUTOR.max_concurrency,
            f"max_running={data[1]} exceeded the cap {_EXECUTOR.max_concurrency}",
        ),
    )[-1],
    input_status=f"{STRESS_COUNT} jobs dispatched back to back",
    assertion=(
        f"all completed; peak simultaneous 'running' <= max_concurrency="
        f"{_EXECUTOR.max_concurrency}; queued state observed"
    ),
)
FINAL_STATES, MAX_RUNNING, MAX_PENDING, PENDING_SEEN, STRESS_MS = STRESS_RESULT
print(
    f"stress summary: {STRESS_COUNT} jobs in {STRESS_MS:.0f} ms | peak running {MAX_RUNNING} | "
    f"peak pending {MAX_PENDING} (queued state observed: {PENDING_SEEN}) | "
    f"executor now holds {len(_EXECUTOR)} entries"
)

## 4.10 Server-side data references

The reference flow is what keeps a 100,000-peak spectrum out of the context
window: `load_spectrum` parses a spectrum and returns an opaque
`ptr:spectrum:...` handle, `summarise_reference` reads it back from server
memory, `search_library` accepts it as the query, and `release_reference`
frees it. The assertions below check both halves of that contract: the
payload is reachable without re-reading the file, and it is **absent** from
the tool response.


In [ ]:
# =============================================================================
# 4.10 Server-side data references — the memory-pointer contract
# =============================================================================
REFERENCE_HOLDER: dict[str, Any] = {}


def load_and_reference() -> Any:
    """Load one spectrum and hold its reference for the later cases."""
    result = sync_call(
        "load_spectrum", file_path=str(EXPERIMENTAL_MZML), spectrum_index=0
    )
    REFERENCE_HOLDER["result"] = result
    return result


REF_LOADED = check(
    "load_spectrum",
    "returns a compact server-side reference instead of the peak arrays",
    load_and_reference,
    predicate=lambda result: (
        expect(
            result.reference.startswith("ptr:spectrum:"),
            f"unexpected reference {result.reference!r}",
        ),
        expect(result.kind == "spectrum", f"kind {result.kind!r}"),
        expect(result.backend == "msmcp.mzml", f"backend {result.backend!r}"),
        expect(
            result.n_peaks == len(CAFFEINE_MS2.peaks),
            f"n_peaks {result.n_peaks} != {len(CAFFEINE_MS2.peaks)}",
        ),
        expect(
            result.provenance["operation"] == "load_spectrum",
            "provenance operation missing",
        ),
        expect(
            result.provenance["sources"][0]["backend"] == "msmcp.mzml",
            "provenance source backend missing",
        ),
        expect(
            "intensity" not in result.model_dump() and "mz" not in result.model_dump(),
            "the peak payload leaked into the tool response",
        ),
        expect(
            len(json.dumps(result.model_dump())) < 4096,
            f"response is {len(json.dumps(result.model_dump()))} characters",
        ),
    )[-1],
    input_status=f"{EXPERIMENTAL_MZML.name}, spectrum_index=0",
    assertion=(
        "reference + metadata + provenance returned; payload absent; response < 4 kB"
    ),
)

check(
    "summarise_reference",
    "reads the spectrum back from server memory (no file re-read)",
    lambda: sync_call("summarise_reference", reference=REF_LOADED.reference),
    predicate=lambda text: (
        must_contain(
            text,
            "Spectrum #0",
            "Held server-side",
            "Provenance: load_spectrum",
            str(EXPERIMENTAL_MZML.resolve()),
        ),
    )[-1],
    input_status=f"reference {REF_LOADED.reference[:24]}…",
    assertion="summary rendered from stored peaks; provenance chain back to the file intact",
)


def reference_round_trip() -> tuple[str, str, PollOutcome]:
    """Search using a server-side reference as the query spectrum."""
    started = time.perf_counter()
    text = async_call(
        "search_library",
        spectrum_reference=REF_LOADED.reference,
        database_file=str(LIBRARY_MZML),
        chunk_size=500,
    )
    match = JOB_ID_RE.search(text)
    if match is None:
        raise AssertionError(f"no job id in dispatch payload:\n{text}")
    job_id = match.group(1)
    JOB_STARTED[job_id] = started
    JOB_DISPATCH_MS[job_id] = (time.perf_counter() - started) * 1000.0
    return job_id, text, poll_job(job_id)


check(
    "search_library",
    "searches with a reference as the query, linking provenance to it",
    reference_round_trip,
    predicate=completed_report(
        "SYNTHETIC LIBRARY",
        "held server-side",
        f"reference `{REF_LOADED.reference}`",
        "Provenance: search_library",
    ),
    input_status=f"reference {REF_LOADED.reference[:24]}… + {LIBRARY_MZML.name}",
    assertion="completed; report credits the reference as the real query and carries provenance",
)

check(
    "release_reference",
    "releases the reference and reclaims its memory",
    lambda: sync_call("release_reference", reference=REF_LOADED.reference),
    predicate=lambda text: must_contain(text, "Released"),
    input_status=f"reference {REF_LOADED.reference[:24]}…",
    assertion="released explicitly rather than waiting for the retention period",
)

check(
    "release_reference",
    "releasing an already-released reference is idempotent",
    lambda: sync_call("release_reference", reference=REF_LOADED.reference),
    predicate=lambda text: must_contain(text, "was not held", "No action taken"),
    input_status="same reference, released twice",
    assertion="no error on retry — a lost response cannot cause a spurious failure",
)

check(
    "summarise_reference",
    "a released reference is refused, never silently empty",
    lambda: sync_call("summarise_reference", reference=REF_LOADED.reference),
    raises=UnknownReferenceError,
    predicate=lambda exc: must_contain(str(exc), "No live data reference"),
    input_status="reference released in the previous case",
    assertion="UnknownReferenceError naming the missing reference",
)

check(
    "load_spectrum",
    "an out-of-range spectrum index is refused, not clamped",
    lambda: sync_call(
        "load_spectrum", file_path=str(EXPERIMENTAL_MZML), spectrum_index=99
    ),
    raises=SpectrumIndexError,
    predicate=lambda exc: must_contain(str(exc), "no spectrum at index"),
    input_status=f"{EXPERIMENTAL_MZML.name}, spectrum_index=99 (file has 3)",
    assertion="SpectrumIndexError — a readable file with a bad index fails loudly",
)

In [ ]:
# =============================================================================
# 4.11 Report delivery — the report crosses the wire once, then a digest
# =============================================================================
# The report is the largest payload MSMCP emits, so a client that keeps polling
# a finished job must not keep paying for it: the first terminal poll returns
# the report, every later poll returns a short digest, and `full_report=True`
# re-requests the report for a host that has lost it.
REPORT_BYTES = len(OUTCOME_CLASSICAL.final_payload)
REPEAT_POLLS = [
    async_call("check_search_status", job_id=JOB_CLASSICAL) for _ in range(5)
]

check(
    "check_search_status",
    "repeat polls of a finished job return a digest, not the report",
    lambda: REPEAT_POLLS,
    predicate=lambda polls: (
        must_contain(
            polls[0], "\u2705 **Completed**", "Synthetic library", "full_report=True"
        ),
        expect(
            all("## Spectral Library Search Results" not in poll for poll in polls),
            "a repeat poll returned the full report again",
        ),
        expect(
            all(len(poll) * 3 < REPORT_BYTES * 2 for poll in polls),
            f"a repeat poll cost more than two thirds of the report: "
            f"{[len(poll) for poll in polls]} against {REPORT_BYTES} bytes",
        ),
    )[-1],
    input_status=f"job {JOB_CLASSICAL[:8]}\u2026 polled 5 more times after its report",
    assertion="a digest on every repeat poll, each under two thirds of the report",
)

check(
    "check_search_status",
    "full_report=True re-delivers the report verbatim",
    lambda: async_call("check_search_status", job_id=JOB_CLASSICAL, full_report=True),
    predicate=lambda text: expect(
        text == OUTCOME_CLASSICAL.final_payload,
        f"re-requested report differs ({len(text)} vs {REPORT_BYTES} bytes)",
    ),
    input_status=f"job {JOB_CLASSICAL[:8]}\u2026 with full_report=True",
    assertion="the report stays reproducible on demand after it has been delivered",
)

check(
    "check_search_status",
    "polling a finished job six times costs less than three reports",
    lambda: (REPORT_BYTES, sum(len(poll) for poll in REPEAT_POLLS)),
    predicate=lambda cost: expect(
        cost[1] < 3 * cost[0],
        f"repeat polls cost {cost[1]} bytes against a {cost[0]}-byte report; the "
        f"delivery contract is meant to bound the context cost of polling",
    ),
    input_status="cumulative context cost of polling one finished job",
    assertion="the context cost of repeat polling is bounded by the digest, not the report",
)

## 5. Diagnostic Summary Table

Every `check()` call recorded a row. The frame below reports, per case: the
tool, its input status, the verdict, the runtime in milliseconds, and what was
asserted about the wire schema / contract. Polars is used when installed, then
pandas; otherwise the table renders as Markdown so the notebook still runs in
a minimal environment.

In [ ]:
# =============================================================================
# 5.1 Build the diagnostic frame
# =============================================================================
ROWS = [
    {
        "tool": row.tool,
        "case": row.case,
        "input_status": row.input_status,
        "status": row.status,
        "runtime_ms": round(row.runtime_ms, 3),
        "schema_assertion": row.schema_assertion,
    }
    for row in RESULTS
]
_ORDER = {"FAIL": 0, "SKIP": 1, "PASS": 2}
ROWS.sort(key=lambda r: (_ORDER.get(r["status"], 3), r["tool"], r["case"]))


def frame_from_rows(rows: list[dict[str, Any]]) -> tuple[str, Any]:
    """Build a Polars or pandas frame; fall back to plain rows."""
    try:
        import polars as pl  # type: ignore

        return "polars", pl.DataFrame(rows)
    except ImportError:
        pass
    try:
        import pandas as pd  # type: ignore

        return "pandas", pd.DataFrame(rows)
    except ImportError:
        return "markdown", rows


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    lines = [
        "| " + " | ".join(columns) + " |",
        "|" + "|".join(["---"] * len(columns)) + "|",
    ]
    for row in rows:
        cells = [str(row.get(c, "")).replace("|", "\\|") for c in columns]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def show_table(table: Any, columns: list[str] | None = None) -> None:
    """Render rich output in IPython, print otherwise."""
    resolved = columns or (list(table[0]) if isinstance(table, list) and table else [])
    try:
        # IPython is only present in a notebook kernel, not in the headless runner.
        from IPython.display import (  # pyright: ignore[reportMissingImports]
            Markdown,
            display,
        )

        if isinstance(table, list):
            display(Markdown(markdown_table(table, resolved)))
        else:
            display(table)
        return
    except ImportError:
        pass
    print(markdown_table(table, resolved) if isinstance(table, list) else table)


FRAME_KIND, DIAGNOSTIC_FRAME = frame_from_rows(ROWS)
FRAME_COLUMNS = [
    "tool",
    "case",
    "input_status",
    "status",
    "runtime_ms",
    "schema_assertion",
]
print(f"dataframe backend: {FRAME_KIND} ({len(ROWS)} rows)")
if FRAME_KIND != "markdown":
    show_table(DIAGNOSTIC_FRAME[FRAME_COLUMNS])
else:
    show_table(ROWS, FRAME_COLUMNS)

In [ ]:
# =============================================================================
# 5.2 Per-tool roll-up (pure-Python aggregation — backend independent)
# =============================================================================
_by_tool: dict[str, list[CheckResult]] = defaultdict(list)
for _row in RESULTS:
    _by_tool[_row.tool].append(_row)

agg_rows: list[dict[str, Any]] = []
for _tool, _rows in sorted(_by_tool.items()):
    _runtimes = [r.runtime_ms for r in _rows]
    agg_rows.append(
        {
            "tool": _tool,
            "cases": len(_rows),
            "pass": sum(r.status == "PASS" for r in _rows),
            "fail": sum(r.status == "FAIL" for r in _rows),
            "skip": sum(r.status == "SKIP" for r in _rows),
            "runtime_total_ms": round(sum(_runtimes), 3),
            "runtime_median_ms": round(float(np.median(_runtimes)), 3),
            "runtime_max_ms": round(max(_runtimes), 3),
        }
    )
AGG_COLUMNS = [
    "tool",
    "cases",
    "pass",
    "fail",
    "skip",
    "runtime_total_ms",
    "runtime_median_ms",
    "runtime_max_ms",
]
_agg_kind, AGG_FRAME = frame_from_rows(agg_rows)
show_table(AGG_FRAME[AGG_COLUMNS] if _agg_kind != "markdown" else agg_rows, AGG_COLUMNS)

In [ ]:
# =============================================================================
# 5.3 Failures, skips & final verdict
# =============================================================================
FAILED = [row for row in RESULTS if row.status == "FAIL"]
SKIPPED = [row for row in RESULTS if row.status == "SKIP"]
PASSED = [row for row in RESULTS if row.status == "PASS"]
SUITE_MS = (time.perf_counter() - SUITE_START) * 1000.0

if FAILED:
    print("FAILURES")
    for row in FAILED:
        print(f"\n❌ [{row.tool}] {row.case} :: {row.schema_assertion}")
        print(f"   input: {row.input_status}")
        print("   " + (row.detail or "no detail").replace("\n", "\n   ")[:2000])

if SKIPPED:
    print("\nSKIPPED (environment-dependent)")
    for row in SKIPPED:
        print(f"  ⏭️  [{row.tool}] {row.case} — {row.schema_assertion}")

verdict = "PASSED" if not FAILED else "FAILED"
banner = f"EVAL {verdict}: {len(PASSED)} passed, {len(FAILED)} failed, {len(SKIPPED)} skipped"
print("\n" + "=" * len(banner))
print(banner)
print(f"total wall clock: {SUITE_MS / 1000:.2f} s")
print(f"artifacts       : {ARTIFACT_DIR}")
print("=" * len(banner))

ALL_EVALUATED = set(_by_tool)
UNEXERCISED = (
    EXPECTED_TOOLS
    | {
        "tools/list",
        "embed",
        "get_embedder",
        "DreaMSInferenceEmbedder",
        "LSMMS2InferenceEmbedder",
    }
) - ALL_EVALUATED
print("tools with no evaluated case:", sorted(UNEXERCISED) or "none")